# Systematic Alpha Lab: Research Workflow Demo

This notebook shows what the current code is useful for today.

It runs a complete, credential-free equity research loop:

1. Create a synthetic equity universe with sectors and forward returns.
2. Build simple factors from prices.
3. Clean factors cross-sectionally and sector-neutralize them.
4. Evaluate factor IC, IR, long-short behavior, and coverage.
5. Combine factors into an alpha score.
6. Convert the alpha into a simple top-minus-bottom portfolio test.

The point is not that synthetic data proves an investable strategy. The point is that the package now has reusable research plumbing: data containers, factor cleaning, factor evaluation, alpha combination, and alpha diagnostics.

## 1. Setup

Use the editable package install if available. The path fallback keeps the notebook runnable from VS Code even when the selected kernel is not the same environment used for installation.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("/Users/edl/Documents/dev/quantlab_v2/quantlab_step0_intro")
SRC = PROJECT_ROOT / "src"

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

PROJECT_ROOT

In [ ]:
import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    plt = None

from systematic_alpha_lab.workflows.synthetic_equity_alpha import (
    make_synthetic_equity_data,
    build_simple_factors,
    evaluate_factors,
    build_alpha,
    run_synthetic_equity_alpha_demo,
)

if plt is not None:
    plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## 2. Run The End-To-End Workflow

`run_synthetic_equity_alpha_demo` is the simplest public entry point. Internally it calls the same modular pieces shown later in the notebook.

In [ ]:
data, factor_result, alpha_result = run_synthetic_equity_alpha_demo(
    n_days=504,
    n_assets=80,
    seed=7,
)

type(data), type(factor_result), type(alpha_result)

## 3. Inspect The Data Artifact

The workflow returns a `DataBundle`: prices, forward returns, sector labels, and metadata. This is the first useful design choice because downstream code can consume one explicit object instead of many loose files.

In [ ]:
summary = pd.Series({
    "price_rows": data.prices.shape[0],
    "assets": data.prices.shape[1],
    "sectors": data.sectors.nunique(),
    "first_date": data.prices.index.min().date(),
    "last_date": data.prices.index.max().date(),
    "seed": data.metadata["seed"],
})
summary

In [ ]:
data.prices.iloc[:5, :8]

In [ ]:
sector_counts = data.sectors.value_counts().sort_index()
sector_counts

## 4. Build And Inspect Factors

`build_simple_factors` creates three cross-sectional signals from prices:

- `momentum_21d`: recent 21-day return.
- `reversal_5d`: negative recent 5-day return.
- `low_vol_21d`: negative 21-day realized volatility.

Each factor is then cleaned using the existing factor research layer: sector neutralization, winsorization, and cross-sectional standardization.

In [ ]:
factors = build_simple_factors(data)
list(factors)

In [ ]:
factor_snapshot_date = data.prices.index[80]
pd.DataFrame({name: factor.loc[factor_snapshot_date] for name, factor in factors.items()}).head(10)

In [ ]:
coverage = pd.Series({name: factor.notna().mean().mean() for name, factor in factors.items()})
coverage.to_frame("average_non_null_coverage")

## 5. Evaluate The Factors

`evaluate_factors` calls the factor analytics layer and returns a `FactorResearchResult`. This is the first research screen: which individual signals have evidence, how stable are they, and did the cleaning leave enough usable observations?

In [ ]:
factor_result = evaluate_factors(data, factors)
factor_analytics = pd.DataFrame(factor_result.analytics).T
factor_analytics

In [ ]:
factor_plot_data = factor_analytics[["mean_ic", "ic_ir", "ls_sharpe"]]

if plt is not None:
    ax = factor_plot_data.plot(kind="bar", figsize=(9, 4))
    ax.axhline(0, color="black", linewidth=1)
    ax.set_title("Factor Evaluation Summary")
    ax.set_ylabel("Metric value")
    plt.xticks(rotation=0)
    plt.tight_layout()
    factor_plot_output = ax
else:
    factor_plot_output = factor_plot_data

factor_plot_output

## 6. Build A Composite Alpha

`build_alpha` combines cleaned factor scores into one alpha matrix. In the current code the weights are deliberately simple:

- momentum: 50%
- reversal: 25%
- low volatility: 25%

This gives the next layers a clean input: one date-by-asset alpha score panel.

In [ ]:
alpha = build_alpha(factors)

alpha_summary = pd.Series({
    "rows": alpha.shape[0],
    "assets": alpha.shape[1],
    "avg_coverage": alpha.notna().mean().mean(),
    "avg_cross_sectional_mean": alpha.mean(axis=1).mean(),
    "avg_cross_sectional_std": alpha.std(axis=1).mean(),
})
alpha_summary

In [ ]:
alpha_cross_section = alpha.loc[factor_snapshot_date].sort_values()

if plt is not None:
    ax = alpha_cross_section.plot(kind="bar", figsize=(12, 3), width=1.0)
    ax.set_title(f"Alpha Cross-Section On {factor_snapshot_date.date()}")
    ax.set_xlabel("Assets sorted by alpha score")
    ax.set_ylabel("Alpha score")
    plt.xticks([])
    plt.tight_layout()
    alpha_cross_section_output = ax
else:
    alpha_cross_section_output = pd.concat([
        alpha_cross_section.head(5).rename("lowest_alpha"),
        alpha_cross_section.tail(5).rename("highest_alpha"),
    ], axis=1)

alpha_cross_section_output

## 7. Evaluate The Composite Alpha

`AlphaResearchResult` stores the alpha score panel, alpha diagnostics, and metadata. The current implemented metrics are IC, IR, turnover, and decay.

In [ ]:
alpha_result = alpha_result

alpha_metrics = pd.Series({
    "ic": alpha_result.metrics["ic"],
    "ir": alpha_result.metrics["ir"],
    "turnover": alpha_result.metrics["turnover"],
    **{f"decay_{h}d": v for h, v in alpha_result.metrics["decay"].items()},
})
alpha_metrics.to_frame("value")

In [ ]:
alpha_result.metadata

## 8. Turn Alpha Scores Into A Simple Portfolio Test

This is a notebook-level demonstration of how the current alpha output becomes useful for portfolio research. It is intentionally simple: each day, buy the top quintile by alpha, short the bottom quintile, equal-weight both sides, then observe next-day returns.

The formal risk model and portfolio construction modules should replace this simple test next.

In [ ]:
def top_bottom_long_short_returns(alpha_scores: pd.DataFrame, forward_returns: pd.DataFrame, quantile: float = 0.2) -> pd.Series:
    portfolio_returns = []
    dates = []

    for dt in alpha_scores.index.intersection(forward_returns.index):
        scores = alpha_scores.loc[dt].dropna()
        rets = forward_returns.loc[dt].reindex(scores.index).dropna()
        scores = scores.reindex(rets.index).dropna()

        if len(scores) < 10:
            continue

        n = max(int(len(scores) * quantile), 1)
        longs = scores.nlargest(n).index
        shorts = scores.nsmallest(n).index
        portfolio_returns.append(rets.loc[longs].mean() - rets.loc[shorts].mean())
        dates.append(dt)

    return pd.Series(portfolio_returns, index=pd.DatetimeIndex(dates), name="long_short_return")


ls_returns = top_bottom_long_short_returns(alpha, data.returns_forward)
ls_returns.describe()

In [ ]:
performance = pd.Series({
    "annualized_return": ls_returns.mean() * 252,
    "annualized_volatility": ls_returns.std() * np.sqrt(252),
    "sharpe": (ls_returns.mean() / ls_returns.std()) * np.sqrt(252),
    "hit_rate": (ls_returns > 0).mean(),
    "observations": len(ls_returns),
})
performance.to_frame("value")

In [ ]:
cumulative = (1 + ls_returns.fillna(0)).cumprod() - 1

if plt is not None:
    fig, ax = plt.subplots(figsize=(10, 4))
    cumulative.plot(ax=ax)
    ax.set_title("Toy Top-Minus-Bottom Alpha Portfolio")
    ax.set_ylabel("Cumulative return")
    ax.set_xlabel("Date")
    plt.tight_layout()
    cumulative_output = ax
else:
    cumulative_output = cumulative.tail(10).to_frame("cumulative_return")

cumulative_output

## 9. What This Code Is Useful For Now

The current package is useful as a research scaffold:

- `DataBundle` makes the research input explicit: prices, returns, sectors, metadata.
- `build_simple_factors` shows the expected factor interface: one date-by-asset score matrix per signal.
- `clean_factor` standardizes and sector-neutralizes raw factors.
- `evaluate_factors` produces IC, IR, long-short, and coverage diagnostics.
- `build_alpha` converts multiple cleaned signals into one alpha score panel.
- `evaluate_alpha` measures whether the composite alpha has predictive content and how quickly it decays.

The next valuable extension is the risk model. After this notebook creates an alpha panel, the next module should estimate asset risk and factor/sector exposures. Portfolio construction should consume both `alpha` and `risk`, instead of directly sorting alpha into a naive top-minus-bottom portfolio.

## 10. Where Each Piece Lives

- Workflow: `src/systematic_alpha_lab/workflows/synthetic_equity_alpha.py`
- Research artifacts: `src/systematic_alpha_lab/core/artifacts.py`
- Factor cleaning: `src/systematic_alpha_lab/factor_research/transforms.py`
- Factor analytics: `src/systematic_alpha_lab/factor_research/analytics.py`
- Alpha diagnostics: `src/systematic_alpha_lab/alpha/evaluate.py`
- Future layer: `src/systematic_alpha_lab/risk/`
- Future layer: `src/systematic_alpha_lab/portfolio/`